# Практика 2. Разведочный анализ и пайплайн — версия с решениями

> Этот файл предназначен преподавателю: все задания заполнены, добавлены комментарии к разбору. Студентам выдаётся версия без суффикса `_solutions`.

# Практика 2. Разведочный анализ и пайплайн

**Курс «Машинное обучение» · Часть 1, занятие 2**

План:

1. Разведочный анализ по порядку: обзор → одномерный → двумерный → связь с целевой
2. Пропуски и выбросы
3. Конструирование признаков
4. **Утечка данных** — увидим её своими глазами на живом эксперименте
5. Сборка пайплайна, который от утечки защищает

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

df = sns.load_dataset("titanic")

print(df.shape)
df.head(3)

> **Что мы сегодня берём авансом.** В этом занятии несколько раз
> встретятся вещи из будущих лекций: логистическая регрессия (лекция 3),
> кросс-валидация и метрика ROC-AUC (лекция 4), метод ближайших
> соседей (лекция 5).
> Сегодня они нужны как **измерительный прибор**: чтобы сказать, помог
> признак или нет, качество нужно чем-то измерить. Как эти модели устроены
> внутри — разберём в своё время, сейчас достаточно относиться к ним как
> к чёрному ящику, который выдаёт число.

---
## 1. Обзор

Первый шаг всегда одинаковый: понять, из чего состоит таблица.

In [ ]:
overview = pd.DataFrame({
    "тип": df.dtypes.astype(str),
    "уникальных": df.nunique(),
    "пропусков": df.isna().sum(),
    "доля пропусков, %": (df.isna().mean() * 100).round(1),
})
display(overview)

Уже видно кое-что важное: столбцы `alive`, `class`, `who`, `adult_male`, `embark_town`, `alone` — это производные от других столбцов. `alive` вообще дублирует целевую переменную.

**В реальных данных такие дубликаты — источник утечки.** Их нужно найти и убрать до обучения.

In [ ]:
# Проверим догадку: alive и survived — одно и то же
print(pd.crosstab(df["survived"], df["alive"]))
print()
print("class дублирует pclass:")
print(pd.crosstab(df["pclass"], df["class"]))

LEAKY = ["alive", "class", "who", "adult_male", "embark_town", "alone", "deck"]
print(f"\nИсключаем из работы: {LEAKY}")

---
## 2. Одномерный анализ

Смотрим на каждый признак по отдельности.

In [ ]:
num_features = ["age", "fare", "sibsp", "parch"]

fig, axes = plt.subplots(2, len(num_features), figsize=(15, 7))
for i, col in enumerate(num_features):
    sns.histplot(df[col].dropna(), bins=30, ax=axes[0, i], color="#2B5CE6")
    axes[0, i].set_title(col)
    sns.boxplot(y=df[col], ax=axes[1, i], color="#C9D6F7")
plt.tight_layout()
plt.show()

Что видно:

- **age** — распределение близко к симметричному, но есть заметный горб у детей
- **fare** — классический длинный хвост вправо: большинство билетов дешёвые, единицы очень дорогие
- **sibsp** и **parch** — по сути счётчики, большинство значений нулевые

In [ ]:
# Длинный хвост лечится логарифмом
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["fare"], bins=40, ax=axes[0], color="#2B5CE6")
axes[0].set_title("fare — исходный")
sns.histplot(np.log1p(df["fare"]), bins=40, ax=axes[1], color="#1B8A5A")
axes[1].set_title("log(1 + fare)")
plt.tight_layout()
plt.show()

print(f"асимметрия до:    {df['fare'].skew():.2f}")
print(f"асимметрия после: {np.log1p(df['fare']).skew():.2f}")

> **Для преподавателя.** log1p, а не log: в данных есть нулевые значения fare (члены команды и приглашённые), и обычный логарифм дал бы минус бесконечность.

### Задание 1

Изучите признак `age`:
1. Постройте гистограмму отдельно для выживших и погибших на одном графике
2. Посчитайте долю детей (младше 16) среди выживших и среди погибших
3. Сделайте вывод: помогает ли возраст различать классы?

Сохраните в переменные: `n_children` — число детей младше 16, `survival_children` и `survival_adults` — доли выживших среди детей и среди взрослых.

**Важно:** возраст известен не у всех — сравнивайте только те строки, где он заполнен. Иначе 177 пассажиров с пропуском попадут во взрослых и исказят их долю.

> **Что посмотреть:** `sns.histplot` — аргументы `data`, `x`, `hue` (разделяет цветом по столбцу), `bins`, `alpha`. Строки с известным возрастом отбираются через `df[df["age"].notna()]`. Осторожно со сравнением: `NaN < 16` даёт `False`, поэтому пропуски сами собой попадают в группу «не дети».

In [ ]:
# 1. Наложенные распределения
plt.figure(figsize=(10, 4))
sns.histplot(data=df, x="age", hue="survived", bins=30,
             palette=["#C0393B", "#1B8A5A"], alpha=0.6)
plt.title("Возраст: выжившие и погибшие")
plt.show()

# 2. Доля детей в каждой группе
children = df["age"] < 16
tab = pd.crosstab(df["survived"], children, normalize="index")
tab.columns = ["взрослые", "дети"]
print("доля детей внутри каждой группы:")
display((tab * 100).round(1))

# Обратный взгляд: выживаемость среди детей и взрослых
print("\nдоля выживших:")
display(df.assign(child=children).groupby("child")["survived"]
          .agg(["mean", "count"]).round(3))

print()
print("Вывод: у детей выживаемость заметно выше — правило «женщины и дети")
print("вперёд» видно прямо в данных. Но в целом распределения возраста")
print("сильно перекрываются, поэтому сам по себе возраст — слабый признак.")
print("Полезнее оказывается производный бинарный признак «ребёнок».")

# Величины для проверки. Считаем ТОЛЬКО по строкам с известным возрастом:
# сравнение «дети против взрослых» осмысленно лишь там, где возраст есть
known = df[df["age"].notna()]
by_child = known.groupby(known["age"] < 16)["survived"].mean()

n_children = (known["age"] < 16).sum()
survival_children = by_child.loc[True]
survival_adults = by_child.loc[False]

print(f"\nдетей младше 16:              {n_children}")
print(f"выживаемость детей:           {survival_children:.4f}")
print(f"выживаемость взрослых:        {survival_adults:.4f}")

# А теперь — почему пропуски нельзя просто оставить в группе взрослых
unknown_rate = df.loc[df["age"].isna(), "survived"].mean()
print(f"\nвыживаемость тех, у кого возраст НЕ УКАЗАН: {unknown_rate:.4f}")
print(f"их в выборке: {df['age'].isna().sum()}")
print("Если бы мы оставили их во взрослых, доля выживших взрослых упала бы")
print("с 0.38 до 0.36 — и мы бы приписали это возрасту, а не пропуску.")

In [ ]:
# Проверка задания 1
assert int(n_children) == 83, f"n_children = {n_children}"
assert round(float(survival_children), 2) == 0.59, f"survival_children = {survival_children}"
assert round(float(survival_adults), 2) == 0.38, f"survival_adults = {survival_adults}"
assert survival_children > survival_adults, "у детей выживаемость должна быть выше"
print("Задание 1 — верно ✓")

> **Для преподавателя.** Здесь стоит подчеркнуть: сырой признак может быть слабым, а производный от него — сильным. Это и есть суть конструирования признаков.

In [ ]:
# Категориальные признаки
cat_features = ["sex", "pclass", "embarked"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, cat_features):
    sns.countplot(data=df, x=col, ax=ax, color="#2B5CE6")
    ax.set_title(col)
plt.tight_layout()
plt.show()

for col in cat_features:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).to_string())

---
## 3. Связь с целевой переменной

Главная часть анализа: ищем то, на что модель сможет опереться.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cat_features):
    rates = df.groupby(col, observed=True)["survived"].mean().sort_values()
    rates.plot(kind="bar", ax=ax, color="#2B5CE6")
    ax.axhline(df["survived"].mean(), color="#C0393B", linestyle="--",
               label="средняя по выборке")
    ax.set_title(f"выживаемость по {col}")
    ax.set_ylabel("доля выживших")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Взаимодействие двух признаков часто сильнее каждого по отдельности
pivot = df.pivot_table(values="survived", index="sex",
                       columns="pclass", aggfunc="mean")

plt.figure(figsize=(7, 3.5))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn", center=0.5,
            vmin=0, vmax=1, linewidths=1)
plt.title("Доля выживших: пол и класс каюты")
plt.show()

print("Женщины первого класса: 97% выживших. Мужчины третьего: 14%.")
print("Разница в семь раз — это и есть взаимодействие признаков.")

### Задание 2

Проверьте признак `embarked` (порт посадки):
1. Постройте выживаемость по портам
2. Проверьте, не объясняется ли разница составом пассажиров по классам
3. Сделайте вывод: `embarked` несёт собственную информацию или он лишь отражение класса?

Сохраните: `survival_port_c` — выживаемость севших в Шербуре (`C`), `class1_share_c` — доля пассажиров первого класса среди них, `survival_c_third` — выживаемость в порту `C` среди пассажиров третьего класса.

> **Что посмотреть:** `df.groupby("embarked")["survived"].agg(["mean", "count"])`, `pd.crosstab(..., normalize="index")` для состава групп и `df.pivot_table(values=, index=, columns=, aggfunc="mean")` — она разворачивает вторую группировку в столбцы.

In [ ]:
# 1. Выживаемость по портам
print("выживаемость по портам:")
display(df.groupby("embarked")["survived"].agg(["mean", "count"]).round(3))

# 2. Состав пассажиров по классам внутри каждого порта
print("\nраспределение классов внутри порта (доли):")
display(pd.crosstab(df["embarked"], df["pclass"], normalize="index").round(3))

# 3. Выживаемость по портам ВНУТРИ каждого класса
print("\nвыживаемость по портам внутри класса:")
display(df.pivot_table(values="survived", index="embarked",
                       columns="pclass", aggfunc="mean").round(3))

print()
print("Вывод: у порта C выживаемость выше — но оттуда садилось больше")
print("пассажиров первого класса. Внутри одного класса разница между")
print("портами заметно меньше. Значит embarked во многом — отражение")
print("класса, а не самостоятельный фактор. Признак стоит оставить,")
print("но не ждать от него многого.")

# Величины для проверки
survival_port_c = df.groupby("embarked")["survived"].mean().loc["C"]
class1_share_c = pd.crosstab(df["embarked"], df["pclass"],
                             normalize="index").loc["C", 1]
survival_c_third = df.pivot_table(values="survived", index="embarked",
                                  columns="pclass", aggfunc="mean").loc["C", 3]

In [ ]:
# Проверка задания 2
assert round(float(survival_port_c), 2) == 0.55, f"survival_port_c = {survival_port_c}"
assert round(float(class1_share_c), 2) == 0.51, f"class1_share_c = {class1_share_c}"
assert round(float(survival_c_third), 2) == 0.38, f"survival_c_third = {survival_c_third}"
print("Задание 2 — верно ✓")

> **Для преподавателя.** Это пример смешивающего фактора (confounder). Разница по портам почти полностью объясняется третьей переменной. Полезно связать с тем, что говорилось на лекции про корреляцию и причинность.

In [ ]:
# Полный обзор двумерных связей: числовые признаки против целевой
num_cols = ["age", "fare", "sibsp", "parch", "pclass"]

fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 3.6))
for ax, col in zip(axes, num_cols):
    for cls, color, label in [(0, "#C0393B", "погиб"), (1, "#1B8A5A", "выжил")]:
        sns.kdeplot(df.loc[df["survived"] == cls, col].dropna(),
                    ax=ax, color=color, fill=True, alpha=0.35, label=label)
    ax.set_title(col)
    ax.set_ylabel("")
axes[0].legend(fontsize=9)
plt.suptitle("Распределения признаков по классам", y=1.04)
plt.tight_layout()
plt.show()

Чем сильнее разъезжаются красная и зелёная кривые, тем полезнее признак. Полностью совпадающие кривые означают, что признак сам по себе классы не различает.

### Вопрос

Судя по графикам, какие два признака различают классы лучше всего, а какой почти бесполезен? Как бы вы проверили это числом, а не на глаз?

> Подсказка: Один из способов — обучить модель на единственном признаке и посмотреть на её качество.

**Что хотелось услышать:**

Лучше всего различают pclass и fare — их распределения заметно смещены. Хуже всего sibsp и parch: кривые почти совпадают. Проверить числом можно несколькими способами: посчитать разницу средних в единицах стандартного отклонения (величина эффекта), посмотреть на ROC-AUC каждого признака по отдельности как одномерного классификатора, или обучить модель на одном признаке и сравнить с baseline. Последний способ самый честный: он сразу показывает, что признак даёт в терминах итоговой метрики.

In [ ]:
# Проверяем: обучаем модель на каждом признаке по отдельности
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as Pipe
from sklearn.preprocessing import StandardScaler as Scaler

single = {}
for col in num_cols:
    pipe = Pipe([("imp", SimpleImputer(strategy="median")),
                 ("sc", Scaler()),
                 ("clf", LogisticRegression(max_iter=1000))])
    score = cross_val_score(pipe, df[[col]], df["survived"],
                            cv=5, scoring="roc_auc").mean()
    single[col] = score

for col, sc in sorted(single.items(), key=lambda kv: -kv[1]):
    bar = "#" * int((sc - 0.5) * 100)
    print(f"{col:8} ROC-AUC {sc:.3f}  {bar}")
print()
print("Значение 0.5 означает, что признак не несёт информации о классе.")

### Задание 3

Изучите связь категориальных признаков с целевой:

1. Постройте таблицу сопряжённости для `sex` и `survived` — в долях
2. Сделайте то же для `pclass` и `embarked`
3. Найдите категорию с наибольшим отклонением от средней выживаемости
4. Ответьте: какой из трёх признаков вы бы взяли первым, если бы можно было взять только один?

Сохраните: `base_rate` — средняя выживаемость по выборке, `survival_female` и `survival_male` — выживаемость женщин и мужчин, `top_feature` — название признака с самой «говорящей» категорией.

> **Что посмотреть:** `df.groupby(col, observed=True)["survived"].agg(["mean", "size"])` — размер группы смотреть обязательно, иначе легко сделать вывод по трём наблюдениям. Отклонение от среднего по выборке — обычное вычитание, а «самую говорящую» категорию ищут по `abs(отклонения)`.

In [ ]:
base_rate = df["survived"].mean()
print(f"средняя выживаемость по выборке: {base_rate:.3f}\n")

for col in ["sex", "pclass", "embarked"]:
    tab = df.groupby(col, observed=True)["survived"].agg(["mean", "size"])
    tab["отклонение"] = (tab["mean"] - base_rate).round(3)
    tab = tab.round(3)
    print(f"--- {col} ---")
    display(tab)

# Категория с максимальным отклонением
rows = []
for col in ["sex", "pclass", "embarked"]:
    g = df.groupby(col, observed=True)["survived"].agg(["mean", "size"])
    for cat, row in g.iterrows():
        rows.append({"признак": col, "категория": cat,
                     "выживаемость": row["mean"],
                     "отклонение": abs(row["mean"] - base_rate),
                     "объектов": int(row["size"])})
best = pd.DataFrame(rows).sort_values("отклонение", ascending=False)
print("\nсамые «говорящие» категории:")
display(best.head(4).round(3))

print()
print("Какой признак взять первым: sex.")
print("Обоснование: он даёт максимальный разброс выживаемости между")
print("категориями (0.19 против 0.74), при этом обе группы большие —")
print("значит, оценка надёжна. У embarked разброс меньше, и мы уже")
print("выяснили, что он во многом отражает класс, а не несёт своей")
print("информации.")

# Величины для проверки
base_rate = df["survived"].mean()
by_sex = df.groupby("sex")["survived"].mean()
survival_female = by_sex.loc["female"]
survival_male = by_sex.loc["male"]
top_feature = best.iloc[0]["признак"]

In [ ]:
# Проверка задания 3
assert round(float(base_rate), 2) == 0.38, f"base_rate = {base_rate}"
assert round(float(survival_female), 2) == 0.74, f"survival_female = {survival_female}"
assert round(float(survival_male), 2) == 0.19, f"survival_male = {survival_male}"
assert top_feature == "sex", f"top_feature = {top_feature}"
print("Задание 3 — верно ✓")

### Вопрос

Признак `embarked` показал заметную связь с выживаемостью. Но раньше мы выяснили, что она объясняется составом пассажиров по классам. Как называется такая ситуация и чем она опасна при интерпретации модели?

**Что хотелось услышать:**

Это смешивающий фактор (confounder): переменная, которая влияет и на предполагаемую причину, и на следствие, создавая видимость связи между ними. Опасность в том, что модель честно использует embarked для предсказания и он получит ненулевую важность — а мы сделаем вывод, что порт посадки влияет на выживаемость. Для предсказания это не страшно: признак работает. Но для выводов о причинах — категорически нельзя. Именно поэтому важность признаков в модели не отвечает на вопрос «что на что влияет».

---
## 4. Пропуски и выбросы

In [ ]:
# Проверим гипотезу: пропуск возраста — сам по себе информативен?
age_missing = df["age"].isna()
print(f"пропусков возраста: {age_missing.sum()} ({age_missing.mean():.1%})")
print()
print("выживаемость в зависимости от того, известен ли возраст:")
display(df.assign(age_na=age_missing).groupby("age_na")["survived"]
          .agg(["mean", "count"]).round(3))
print()
print("Разница есть — значит флаг пропуска стоит сохранить как признак.")

In [ ]:
# Выбросы по правилу межквартильного размаха
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

low, high = iqr_bounds(df["fare"])
outliers = (df["fare"] < low) | (df["fare"] > high)

print(f"границы по IQR: [{low:.1f}, {high:.1f}]")
print(f"выбросов: {outliers.sum()} ({outliers.mean():.1%})")
print()
print("выживаемость среди выбросов и остальных:")
display(df.assign(out=outliers).groupby("out")["survived"]
          .agg(["mean", "count"]).round(3))
print()
print("Это не ошибки данных, а реальные дорогие билеты первого класса.")
print("Удалять их нельзя — потеряем самую платёжеспособную группу.")

In [ ]:
# Стратегии заполнения пропусков: сравниваем на качестве модели
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

X_imp = df[["age", "fare", "sibsp", "parch", "pclass"]]
y_imp = df["survived"]

strategies = {
    "медиана": SimpleImputer(strategy="median"),
    "среднее": SimpleImputer(strategy="mean"),
    "константа -1": SimpleImputer(strategy="constant", fill_value=-1),
    "медиана + флаг": SimpleImputer(strategy="median", add_indicator=True),
    "по соседям": KNNImputer(n_neighbors=5),
    "итеративный": IterativeImputer(random_state=RANDOM_STATE, max_iter=10),
}

print(f"{'стратегия':18} {'ROC-AUC':>9} {'разброс':>9}")
for name, imputer in strategies.items():
    pipe = Pipeline([("imp", imputer), ("sc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000))])
    s = cross_val_score(pipe, X_imp, y_imp, cv=5, scoring="roc_auc")
    print(f"{name:18} {s.mean():9.4f} {s.std():9.4f}")

### Вопрос

Разница между стратегиями заполнения оказалась небольшой. Значит ли это, что можно не задумываться и всегда брать медиану? В каких случаях выбор стратегии станет принципиальным?

**Что хотелось услышать:**

Не значит. Здесь пропусков около 20% в одном признаке, и он не самый важный — поэтому разница мала.

Выбор станет принципиальным в нескольких случаях. Первый — когда пропусков много: при 50% и выше заполнение константой фактически создаёт новый класс объектов, и это может как помочь, так и всё сломать. Второй — когда пропуск неслучаен: если доход не указывают самые богатые, заполнение медианой систематически занижает данные именно там, где важнее всего. Третий — когда признак ключевой: ошибка в его заполнении бьёт по качеству напрямую.

Практический подход: начать с медианы и флага пропуска, а если признак важный и пропусков много — сравнить несколько стратегий, как сделано выше.

In [ ]:
# Проверяем гипотезу «пропуск неслучаен» напрямую
missing_age = df["age"].isna()

print("сравниваем группы: с пропуском возраста и без\n")
comparison = pd.DataFrame({
    "с пропуском": [
        missing_age.sum(),
        df.loc[missing_age, "survived"].mean(),
        df.loc[missing_age, "fare"].median(),
        (df.loc[missing_age, "pclass"] == 3).mean(),
    ],
    "без пропуска": [
        (~missing_age).sum(),
        df.loc[~missing_age, "survived"].mean(),
        df.loc[~missing_age, "fare"].median(),
        (df.loc[~missing_age, "pclass"] == 3).mean(),
    ],
}, index=["объектов", "доля выживших", "медианный тариф", "доля третьего класса"])
display(comparison.round(3))

print()
print("Пропуски возраста сосредоточены в третьем классе, у пассажиров")
print("с дешёвыми билетами и низкой выживаемостью. Это не случайность —")
print("значит, флаг пропуска несёт информацию, и его стоит оставить.")

---
## 5. Конструирование признаков

Здесь знание задачи даёт больше, чем смена алгоритма.

In [ ]:
def add_features(data):
    """Создаёт производные признаки. Функция, а не код на месте:
    так одно и то же преобразование применится и к обучающей, и к тестовой части."""
    out = data.copy()
    out["family_size"] = out["sibsp"] + out["parch"] + 1
    out["is_alone"] = (out["family_size"] == 1).astype(int)
    out["fare_per_person"] = out["fare"] / out["family_size"]
    out["fare_log"] = np.log1p(out["fare"])
    out["is_child"] = (out["age"] < 16).astype(float)
    out["age_missing"] = out["age"].isna().astype(int)
    return out

df_fe = add_features(df)
new_cols = ["family_size", "is_alone", "fare_per_person",
            "fare_log", "is_child", "age_missing"]
display(df_fe[new_cols].head())

print("\nсвязь новых признаков с целевой:")
for col in new_cols:
    corr = df_fe[col].corr(df_fe["survived"])
    print(f"  {col:18} корреляция с survived: {corr:+.3f}")

### Задание 4

Придумайте и постройте ещё два признака. Идеи:
- размер семьи, разбитый на категории: один / небольшая семья / большая семья
- взаимодействие пола и класса одним признаком
- отклонение стоимости билета от средней по своему классу

Проверьте связь каждого нового признака с целевой переменной.

Сохраните: `survival_big_family` — выживаемость больших семей (пять человек и больше), `corr_fare_vs_class` — корреляция отклонения цены от медианы своего класса с выживаемостью, `worst_group` — связка «пол + класс» с самой низкой выживаемостью.

> **Что посмотреть:** `pd.cut(значения, bins=[...], labels=[...])` режет число на категории; `df.groupby("pclass")["fare"].transform("median")` возвращает медиану своей группы **в форме исходной таблицы** — то, что нужно для отклонения; связка двух признаков собирается конкатенацией строк, а число переводится в строку через `.astype(str)`. Корреляция — `Series.corr(другой)`.

In [ ]:
df_fe2 = df_fe.copy()

# 1. Категория размера семьи. Связь с выживаемостью немонотонна:
#    у одиночек и у больших семей шансы ниже, чем у средних
df_fe2["family_type"] = pd.cut(
    df_fe2["family_size"],
    bins=[0, 1, 4, 20],
    labels=["один", "малая", "большая"],
)

# 2. Отклонение цены билета от типичной для своего класса
class_median = df_fe2.groupby("pclass")["fare"].transform("median")
df_fe2["fare_vs_class"] = df_fe2["fare"] - class_median

# 3. Пол и класс одним признаком
df_fe2["sex_pclass"] = df_fe2["sex"] + "_" + df_fe2["pclass"].astype(str)

print("выживаемость по типу семьи:")
display(df_fe2.groupby("family_type", observed=True)["survived"]
        .agg(["mean", "count"]).round(3))

print("\nвыживаемость по связке пол+класс:")
display(df_fe2.groupby("sex_pclass")["survived"]
        .agg(["mean", "count"]).round(3).sort_values("mean"))

print(f"\nкорреляция fare_vs_class с survived: "
      f"{df_fe2['fare_vs_class'].corr(df_fe2['survived']):+.3f}")

print()
print("Признак family_type ловит немонотонную зависимость, которую")
print("линейная модель по family_size увидеть не смогла бы.")

# Величины для проверки
survival_big_family = df_fe2.loc[df_fe2["family_size"] >= 5, "survived"].mean()
corr_fare_vs_class = df_fe2["fare_vs_class"].corr(df_fe2["survived"])
worst_group = df_fe2.groupby("sex_pclass")["survived"].mean().idxmin()

In [ ]:
# Проверка задания 4
assert round(float(survival_big_family), 2) == 0.16,     f"survival_big_family = {survival_big_family}"
assert round(float(corr_fare_vs_class), 2) == 0.15,     f"corr_fare_vs_class = {corr_fare_vs_class}"
assert worst_group == "male_3", f"worst_group = {worst_group}"
print("Задание 4 — верно ✓")

> **Для преподавателя.** Обратите внимание студентов, что fare_vs_class считается через groupby-transform по всем данным. В домашней работе такое надо делать внутри пайплайна, иначе это утечка — статистика классов считается с участием теста.

In [ ]:
# Кодирование категорий: три способа на одном признаке
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

emb = df[["embarked"]].fillna("S")

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe_result = pd.DataFrame(ohe.fit_transform(emb),
                          columns=ohe.get_feature_names_out(["embarked"]))

ord_enc = OrdinalEncoder()
ord_result = pd.DataFrame(ord_enc.fit_transform(emb), columns=["embarked_ord"])

freq = emb["embarked"].map(emb["embarked"].value_counts(normalize=True))

combined = pd.concat([emb.reset_index(drop=True), ohe_result,
                      ord_result, freq.rename("embarked_freq")], axis=1)
display(combined.head(8).round(3))

### Вопрос

Ordinal-кодирование присвоило портам номера 0, 1, 2. Почему для логистической регрессии это плохой выбор?

**Что хотелось услышать:**

Линейная модель умножает значение признака на вес и складывает. Она интерпретирует номера как величины: порт с номером 2 «вдвое больше» порта с номером 1, а порт 1 оказывается «между» портами 0 и 2. Для названий портов такой порядок бессмысленен, и модель будет вынуждена подгонять один общий вес под три несопоставимые категории.

Поэтому линейным моделям нужен one-hot: каждая категория получает свой вес, и никакого порядка между ними не подразумевается. Ordinal осмыслен там, где порядок настоящий: класс каюты, уровень образования, размер одежды.

In [ ]:
# Проверим утверждение экспериментом: одна и та же модель, два кодирования
y = df["survived"]

for enc_name, X_enc in [("ordinal", ord_result), ("one-hot", ohe_result)]:
    score = cross_val_score(LogisticRegression(max_iter=1000), X_enc, y,
                            cv=5, scoring="roc_auc").mean()
    print(f"{enc_name:10} ROC-AUC = {score:.4f}")

print()
print("One-hot заметно лучше: линейной модели не приходится подгонять")
print("один вес под три категории, выстроенные в выдуманный порядок.")

### Задание 5

Реализуйте target encoding **без утечки**:

1. Напишите функцию, которая кодирует категорию средним значением целевой переменной, но считает это среднее **out-of-fold** — то есть для каждого объекта по данным других фолдов
2. Сравните с наивным вариантом, где среднее считается по всем данным сразу
3. Оцените обе версии на кросс-валидации и объясните разницу

Сохраните ROC-AUC обоих вариантов в `auc_naive` и `auc_oof`, а на искусственном признаке с четырьмя сотнями категорий — в `auc_naive_fake` и `auc_oof_fake` (этот эксперимент идёт в следующей ячейке).

> **Что посмотреть:** разбиение на фолды уже написано за вас, ваша часть — внутри цикла. Пригодятся `Series.iloc[позиции]` для выбора по номерам, `target.groupby(series).mean()` для средних по категориям и `Series.map(словарь_или_Series)` для подстановки. Качество — `cross_val_score(модель, X, y, cv=5, scoring="roc_auc")`.

In [ ]:
from sklearn.model_selection import KFold

def target_encode_naive(series, target):
    """Наивный вариант: среднее по всем данным. Течёт."""
    means = target.groupby(series).mean()
    return series.map(means)


def target_encode_oof(series, target, n_splits=5, random_state=42):
    """Честный вариант: для каждого объекта среднее считается
    по остальным фолдам, без его собственного ответа."""
    encoded = pd.Series(index=series.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    global_mean = target.mean()

    for fit_idx, apply_idx in kf.split(series):
        # статистику считаем ТОЛЬКО по обучающей части фолда
        means = target.iloc[fit_idx].groupby(series.iloc[fit_idx]).mean()
        encoded.iloc[apply_idx] = series.iloc[apply_idx].map(means)

    # категории, не встретившиеся в обучающей части, заполняем общим средним
    return encoded.fillna(global_mean)


cat = df["embarked"].fillna("S")
naive = target_encode_naive(cat, y)
oof = target_encode_oof(cat, y)

print("значения кодирования по портам:")
display(pd.DataFrame({
    "порт": cat, "наивно": naive.round(3), "out-of-fold": oof.round(3)
}).drop_duplicates("порт").head())

# Сравниваем качество
scores_te = {}
for name, enc in [("наивный", naive), ("out-of-fold", oof)]:
    scores_te[name] = cross_val_score(LogisticRegression(max_iter=1000),
                                      enc.to_frame(), y, cv=5,
                                      scoring="roc_auc").mean()
    print(f"{name:14} ROC-AUC = {scores_te[name]:.4f}")

auc_naive, auc_oof = scores_te["наивный"], scores_te["out-of-fold"]

print()
print("На признаке с тремя категориями разница невелика: среднее по сотням")
print("объектов почти не меняется от исключения одного. Утечка target encoding")
print("становится разрушительной, когда категорий много, а объектов в каждой")
print("мало — в пределе, когда категория уникальна для объекта, наивное")
print("кодирование просто подставляет ответ.")

In [ ]:
# Проверка задания 5
assert round(float(auc_naive), 2) == 0.57, f"auc_naive = {auc_naive}"
assert round(float(auc_oof), 2) == 0.55, f"auc_oof = {auc_oof}"
assert abs(auc_naive - auc_oof) < 0.05,     "на признаке с тремя категориями разница должна быть небольшой"
print("Задание 5 — верно ✓")

In [ ]:
# Показываем разрушительный случай: категория почти уникальна
rng = np.random.default_rng(RANDOM_STATE)
fake_cat = pd.Series(rng.integers(0, 400, size=len(df)).astype(str),
                     index=df.index)   # 400 категорий на 900 объектов

naive_fake = target_encode_naive(fake_cat, y)
oof_fake = target_encode_oof(fake_cat, y)

scores_fake = {}
for name, enc in [("наивный", naive_fake), ("out-of-fold", oof_fake)]:
    scores_fake[name] = cross_val_score(LogisticRegression(max_iter=1000),
                                        enc.to_frame(), y, cv=5,
                                        scoring="roc_auc").mean()
    print(f"{name:14} ROC-AUC = {scores_fake[name]:.4f}")

auc_naive_fake = scores_fake["наивный"]
auc_oof_fake = scores_fake["out-of-fold"]

print()
print("Категория случайная и не несёт никакой информации о выживании.")
print("Честное кодирование это показывает — качество на уровне 0.5.")
print("Наивное выдаёт высокий результат, потому что подставляет объекту")
print("его собственный ответ. В продакшене такая модель не сработает.")

In [ ]:
# Проверка: утечка видна на признаке с сотнями категорий
assert round(float(auc_naive_fake), 2) == 0.84, f"auc_naive_fake = {auc_naive_fake}"
assert round(float(auc_oof_fake), 2) == 0.54, f"auc_oof_fake = {auc_oof_fake}"
assert auc_naive_fake - auc_oof_fake > 0.2,     "наивное кодирование случайной категории обязано давать заметно больше честного"
print("Эксперимент с утечкой воспроизведён ✓")

> **Для преподавателя.** Этот пример стоит разобрать особенно внимательно: target encoding — самый распространённый источник утечки в соревнованиях и в реальных проектах. Полезно спросить студентов, где ещё в их работе может возникнуть похожая ситуация.

---
## 6. Утечка данных: эксперимент

Сейчас построим модель, которая покажет отличное качество на **чистом шуме**. Это самая наглядная демонстрация утечки, какую можно устроить.

In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline

# Данные — чистый шум. Никакой связи между X и y не существует
rng = np.random.default_rng(RANDOM_STATE)
n_objects, n_features = 200, 5000
X_noise = rng.normal(size=(n_objects, n_features))
y_noise = rng.integers(0, 2, size=n_objects)

print(f"{n_objects} объектов, {n_features} случайных признаков")
print("Связи между признаками и ответом НЕТ по построению.")
print("Честная модель должна показать ROC-AUC около 0.5")

In [ ]:
# ТАК ДЕЛАТЬ НЕЛЬЗЯ: отбираем признаки по всем данным сразу,
# а потом делаем кросс-валидацию
selector = SelectKBest(f_classif, k=20)
X_selected = selector.fit_transform(X_noise, y_noise)   # <-- утечка здесь

scores_leak = cross_val_score(
    LogisticRegression(max_iter=1000), X_selected, y_noise,
    cv=5, scoring="roc_auc",
)
print(f"ROC-AUC с утечкой:  {scores_leak.mean():.3f}")

In [ ]:
# ТАК ПРАВИЛЬНО: отбор признаков внутри пайплайна,
# то есть заново на каждом обучающем фолде
pipe = Pipeline([
    ("select", SelectKBest(f_classif, k=20)),
    ("clf", LogisticRegression(max_iter=1000)),
])

scores_ok = cross_val_score(pipe, X_noise, y_noise, cv=5, scoring="roc_auc")
print(f"ROC-AUC без утечки: {scores_ok.mean():.3f}")
print()
print(f"Разница: {scores_leak.mean() - scores_ok.mean():+.3f}")
print()
print("В первом случае отбор признаков подсмотрел ответы на всей выборке")
print("и выбрал те, что случайно коррелируют с y. Кросс-валидация уже не")
print("могла это исправить: 'лучшие' признаки были найдены с участием")
print("проверочных фолдов.")
print()
print("Во втором случае отбор происходит внутри каждого фолда заново —")
print("и модель честно показывает, что сигнала нет.")

**Это не искусственный пример.** Ровно так выглядит типичная ошибка: отобрали признаки по корреляции на всех данных, потом «честно» сделали кросс-валидацию и получили прекрасный результат, который в продакшене не воспроизвёлся.

То же самое относится к масштабированию, заполнению пропусков, target encoding и любому другому преобразованию, которое **обучается на данных**.

In [ ]:
# Дубликаты — тихий источник завышенных метрик
dup_full = df.duplicated().sum()
key_cols = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
dup_partial = df.duplicated(subset=key_cols).sum()

print(f"полных дубликатов строк:       {dup_full}")
print(f"дубликатов по ключевым полям:  {dup_partial}")
print()

# Показываем, чем опасны дубликаты, на искусственном примере
X_small = df[["pclass", "age", "fare"]].fillna(df[["pclass", "age", "fare"]].median())
y_small = df["survived"]

# Дублируем часть объектов и смешиваем
dup_idx = np.random.default_rng(RANDOM_STATE).choice(len(X_small), 300)
X_dup = pd.concat([X_small, X_small.iloc[dup_idx]], ignore_index=True)
y_dup = pd.concat([y_small, y_small.iloc[dup_idx]], ignore_index=True)

from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=3)

clean = cross_val_score(knn, X_small, y_small, cv=5, scoring="roc_auc").mean()
dirty = cross_val_score(knn, X_dup, y_dup, cv=5, scoring="roc_auc").mean()

print(f"ROC-AUC без дубликатов: {clean:.4f}")
print(f"ROC-AUC с дубликатами:  {dirty:.4f}")
print(f"завышение: {dirty - clean:+.4f}")

### Вопрос

Почему дубликаты завышают оценку именно при кросс-валидации, и почему метод ближайших соседей страдает от них сильнее других?

**Что хотелось услышать:**

При случайном разбиении копии одного объекта попадают в разные фолды: одна в обучающую часть, другая в проверочную. Модель фактически проверяется на объекте, который уже видела, и отвечает по памяти, а не по обобщению.

Метод ближайших соседей страдает сильнее всех, потому что для него «увидеть объект» означает буквально хранить его. Точная копия окажется на нулевом расстоянии и получит правильный ответ гарантированно. Другие модели тоже выигрывают от дубликатов, но не так прямолинейно.

Отсюда практика: проверять дубликаты **до** разбиения. Причём не только полные, но и почти-дубликаты — записи, различающиеся одним полем.

In [ ]:
# Сложные преобразования внутри пайплайна: FunctionTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler

def add_ratios(X):
    """Признаки-отношения. Должны считаться внутри пайплайна,
    иначе при кросс-валидации получим утечку."""
    X = X.copy()
    X["fare_per_age"] = X["fare"] / X["age"].clip(lower=1)
    X["family_per_fare"] = (X["sibsp"] + X["parch"]) / X["fare"].clip(lower=1)
    return X

ratio_pipe = Pipeline([
    ("ratios", FunctionTransformer(add_ratios)),
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

base_cols = ["age", "fare", "sibsp", "parch", "pclass"]
score_ratios = cross_val_score(ratio_pipe, df[base_cols], df["survived"],
                               cv=5, scoring="roc_auc").mean()

plain_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
score_plain = cross_val_score(plain_pipe, df[base_cols], df["survived"],
                              cv=5, scoring="roc_auc").mean()

print(f"без признаков-отношений: {score_plain:.4f}")
print(f"с признаками-отношениями: {score_ratios:.4f}")
print(f"прирост: {score_ratios - score_plain:+.4f}")

`FunctionTransformer` позволяет положить в пайплайн произвольную функцию. Это важно: конструирование признаков, сделанное «руками» до разбиения, может течь — например, если в нём используются агрегаты по всей выборке.

### Вопрос

Признак `fare_per_age` считается по одной строке и не использует другие объекты. Течёт ли такое преобразование? А признак «отклонение тарифа от среднего по классу»?

**Что хотелось услышать:**

Первый не течёт: он вычисляется из значений одного объекта, и никакая информация о других объектах — в том числе о тестовых — в него не попадает. Такие преобразования можно делать и до разбиения.

Второй течёт. «Среднее по классу» считается по выборке, и если считать его по всем данным сразу, в признак попадёт информация из тестовой части. Правильно — считать среднее только по обучающим данным и применять к тестовым, то есть внутри пайплайна как обучаемое преобразование.

Простое правило: если преобразование использует несколько объектов сразу, оно обязано жить внутри пайплайна.

---
## 7. Пайплайн

Собираем всё вместе так, чтобы утечка стала невозможной по построению.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

data = add_features(df)
numeric = ["age", "fare_log", "family_size", "fare_per_person", "pclass"]
categorical = ["sex", "embarked"]
binary = ["is_alone", "age_missing"]

# is_child кладём в таблицу, но в пайплайн пока не берём —
# он понадобится в задании 6
X = data[numeric + categorical + binary + ["is_child"]]
y = data["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=False)),
        ("scale", StandardScaler()),
    ]), numeric),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", drop="first")),
    ]), categorical),
    ("bin", "passthrough", binary),
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
model.fit(X_train, y_train)
test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f"ROC-AUC на кросс-валидации: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"ROC-AUC на тесте:           {test_auc:.3f}")

### Задание 6

Соберите свой вариант пайплайна и сравните с текущим:

1. Замените `StandardScaler` на `RobustScaler` — поможет ли это, учитывая выбросы в `fare`?
2. Добавьте признак `is_child` (не забудьте: в нём есть пропуски там же, где в `age`)
3. Попробуйте `SimpleImputer(add_indicator=True)` вместо ручного флага
4. Сравните все варианты по кросс-валидации и сделайте вывод

Сохраните ROC-AUC вариантов в `auc_standard`, `auc_robust` и `auc_child`.

> **Что посмотреть:** `RobustScaler` из `sklearn.preprocessing` — интерфейс тот же, что у `StandardScaler`. У `SimpleImputer` есть аргумент `add_indicator=True`: он сам добавит столбец-флаг пропуска. Структуру `ColumnTransformer` проще скопировать из ячейки выше и поменять в ней один блок.

In [ ]:
from sklearn.preprocessing import RobustScaler

def evaluate(prep, name, columns):
    pipe = Pipeline([
        ("prep", prep),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])
    s = cross_val_score(pipe, X_train[columns], y_train, cv=5, scoring="roc_auc")
    print(f"{name:34} {s.mean():.4f} ± {s.std():.4f}")
    return s.mean()

base_cols = numeric + categorical + binary

results = {}

# Вариант 1: исходный
results["StandardScaler"] = evaluate(preprocessor, "исходный (StandardScaler)", base_cols)

# Вариант 2: RobustScaler
prep_robust = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", RobustScaler()),
    ]), numeric),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", drop="first")),
    ]), categorical),
    ("bin", "passthrough", binary),
])
results["RobustScaler"] = evaluate(prep_robust, "RobustScaler", base_cols)

# Вариант 3: добавляем is_child и автоматический индикатор пропусков
numeric3 = numeric + ["is_child"]
cols3 = numeric3 + categorical + binary
prep_child = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ]), numeric3),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", drop="first")),
    ]), categorical),
    ("bin", "passthrough", binary),
])
results["is_child + indicator"] = evaluate(prep_child, "is_child + add_indicator", cols3)

print()
best = max(results, key=results.get)
print(f"лучший вариант: {best}")
print()
print("Выводы:")
print("  RobustScaler не дал заметного выигрыша: мы уже взяли логарифм")
print("  от fare, и хвост распределения перестал быть проблемой.")
print("  Признак is_child помогает — он ловит немонотонную зависимость")
print("  выживаемости от возраста, недоступную линейной модели напрямую.")
print("  Все различия невелики: на 700 объектах разброс кросс-валидации")
print("  сопоставим с разницей между вариантами.")

auc_standard = results["StandardScaler"]
auc_robust = results["RobustScaler"]
auc_child = results["is_child + indicator"]

In [ ]:
# Проверка задания 6
assert round(float(auc_standard), 2) == 0.86, f"auc_standard = {auc_standard}"
assert round(float(auc_robust), 2) == 0.86, f"auc_robust = {auc_robust}"
assert round(float(auc_child), 2) == 0.85, f"auc_child = {auc_child}"
assert abs(auc_standard - auc_robust) < 0.02,     "разница между скейлерами должна быть в пределах шума"
print("Задание 6 — верно ✓")

> **Для преподавателя.** Главный вывод задания: различия между вариантами предобработки обычно в пределах шума. Студенты часто тратят много времени на подбор скейлера, тогда как выигрыш даёт конструирование признаков. Стоит проговорить это явно.

---
## Итоги занятия

- Разведочный анализ идёт по порядку, а не хаотично
- Пропуск и выброс — не всегда мусор, часто это информация
- Производный признак бывает сильнее исходного (`is_child` против `age`)
- **Любое преобразование, которое обучается на данных, обязано жить внутри пайплайна**
- Отбор признаков, масштабирование, заполнение пропусков и target encoding — всё это течёт, если делать их до разбиения

## Домашнее задание 1

Выдаётся сегодня. Вы проделаете то же самое на своих данных: разведочный анализ, чистка, конструирование признаков, пайплайн и baseline.